# Copilot Licensed Users — Direct Ingester (Fabric)

End-to-end Lakehouse loader for **Copilot licensed users** that calls Microsoft Graph directly from inside Fabric. Replaces the prior flow:

```
OLD:  PowerShell → CSV → Files/licensed_raw/ → Copilot_Licensed_Users_Loader.ipynb → Delta
NEW:  This notebook (Graph → Delta)
```

**Source endpoint**: `/v1.0/reports/getOffice365ActiveUserDetail(period='D7')` — returns the M365 active-user report. Each row is enriched with `HasCopilot = TRUE` for a matching Microsoft 365 Copilot assigned product, including Microsoft 365 E7. This is a product-assignment snapshot, not verification of enabled Copilot service plans.

**Output**: Lakehouse Delta table `dbo.copilot_licensed_users` (same name + schema as the existing CSV loader, so the PBIT works without changes).

**Permissions**: app registration with `Reports.Read.All` (Application permission, admin-consented).

**Heads-up — masked UPNs**: if the report shows 32-char hex strings instead of real UPNs, an admin needs to untick **Org settings → Reports → "Display concealed user, group, and site names in all reports"** in `admin.microsoft.com`.


## 1. Configuration

Same app reg + secret as the audit-log direct ingester. The output table is schema-qualified (`dbo.`) to work with both schema-enabled and legacy Lakehouses.


In [ ]:
# === CONFIG ===
TENANT_ID     = '<your-tenant-guid>'
CLIENT_ID     = '<your-app-reg-client-id>'
CLIENT_SECRET = '<your-client-secret-value>'

REPORT_PERIOD = 'D7'                                # 'D7', 'D30', 'D90', 'D180' — Microsoft fixed values
OUTPUT_TABLE  = 'dbo.copilot_licensed_users'        # Delta table consumed by the PBIT
WRITE_MODE    = 'overwrite'                         # 'overwrite' for full snapshots

# --- Copilot licence detection -----------------------------------------------
# getOffice365ActiveUserDetail has no Copilot column, so the licence flag is
# derived from the 'Assigned Products' string. SKU display names change over time
# ('Copilot for Microsoft 365' -> 'Microsoft 365 Copilot'), so match a LIST of
# patterns rather than a single literal.
#
# COPILOT_SKU_PATTERNS : case-insensitive substrings, or =PRODUCT for an exact
#                        product token (after whitespace normalization).
# COPILOT_SKU_EXCLUDE  : products to ignore even if they match above - e.g. trials
#                        or maker licences you do not want counted as seats.
#
# Keep intentional overrides when upgrading; append the =E7 entries to opt in.
# Do not use a broad 'E7' or 'COPILOT' substring as entitlement evidence.
COPILOT_SKU_PATTERNS = [
    'MICROSOFT 365 COPILOT',        # current M365 Copilot seat
    'COPILOT FOR MICROSOFT 365',    # previous name for the same SKU
    'M365 COPILOT',
    '=MICROSOFT 365 E7',           # verified product display name
    '=MICROSOFT_365_E7',           # verified skuPartNumber, if present in input
    '=9a18296a-025f-4e37-9ffa-30bf8d1ce775',  # E7 skuId, not a service-plan ID
]
COPILOT_SKU_EXCLUDE = [
    'VIRAL TRIAL',                  # self-service trials are not paid seats
    'TRIAL',
    'COPILOT STUDIO',
    'SECURITY COPILOT',
    'COPILOT CHAT',                 # standalone Chat is not the paid M365 seat
]


## 2. Authenticate to Microsoft Graph

In [ ]:
import requests

def get_graph_token(tenant_id, client_id, client_secret):
    url  = f'https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token'
    body = {
        'client_id':     client_id,
        'scope':         'https://graph.microsoft.com/.default',
        'client_secret': client_secret,
        'grant_type':    'client_credentials',
    }
    r = requests.post(url, data=body, timeout=30)
    r.raise_for_status()
    return r.json()['access_token']

token   = get_graph_token(TENANT_ID, CLIENT_ID, CLIENT_SECRET)
headers = {'Authorization': f'Bearer {token}'}
print('✓ Graph token acquired.')


## 3. Fetch the M365 active user report

Graph's `getOffice365ActiveUserDetail` endpoint returns a **CSV** body (not JSON). We parse it into a Spark DataFrame.


In [ ]:
import csv
from io import StringIO

report_uri = f"https://graph.microsoft.com/v1.0/reports/getOffice365ActiveUserDetail(period='{REPORT_PERIOD}')"
r = requests.get(report_uri, headers=headers, timeout=120)
r.raise_for_status()

# Strip UTF-8 BOM if present, parse as CSV
csv_text = r.text.lstrip('\ufeff')
reader = csv.DictReader(StringIO(csv_text))
if 'Assigned Products' not in (reader.fieldnames or []):
    raise ValueError("Active-user report is missing 'Assigned Products'; refusing to write false licence flags.")
rows = list(reader)
print(f'✓ Fetched {len(rows):,} rows from {report_uri}')
if rows:
    print('Sample columns:', list(rows[0].keys())[:8], '...')


## 4. Derive `HasCopilot` flag

Match individual `Assigned Products` tokens, separated by `+`, comma or semicolon. Exclusions apply to each product, not the whole user: a trial alongside a qualifying paid product does not remove that product's match. Existing substring aliases remain configurable; the E7 entries use exact matching so E70, trial suffixes and unrelated E7-like names do not qualify.

**Evidence**: Microsoft's [licensing reference](https://learn.microsoft.com/en-us/entra/identity/users/licensing-service-plan-reference) and its linked CSV (updated August 19, 2026) map `Microsoft 365 E7` to `MICROSOFT_365_E7` / `9a18296a-025f-4e37-9ffa-30bf8d1ce775`, including `M365_COPILOT_APPS`. The [E3/E5/E7 feature comparison](https://learn.microsoft.com/en-us/microsoft-365/copilot/microsoft-365-copilot-license-feature-overview) confirms E7 includes Copilot without an add-on. Product IDs are accepted only as complete input tokens; this notebook does not query Graph directory licensing.

The usage report does not expose disabled/provisioning service-plan status and can lag assignments. No-Teams/EEA report labels and other aliases are not guessed. Verify an unmatched label against Microsoft licensing evidence before explicitly adding it. Preserve your existing pattern/exclusion overrides; E7 entries are defaults, not a hidden union with custom settings.


In [ ]:
# === derive the Copilot licence flag =========================================
# Reports the SKUs actually seen, so a renamed product is obvious in the log
# rather than silently producing zero licensed users.
from collections import Counter
import re

def _normalise_product(value):
    return ' '.join((value or '').upper().split())


def _product_tokens(products):
    return [part.strip() for part in re.split(r'[+,;]', products or '') if part.strip()]


_pats = [_normalise_product(p) for p in COPILOT_SKU_PATTERNS if p.strip()]
_excl = [_normalise_product(e) for e in COPILOT_SKU_EXCLUDE if e.strip()]


def _is_copilot(products: str):
    """Return (matched, reason) for an 'Assigned Products' string."""
    excluded = ''
    for product in _product_tokens(products):
        up = _normalise_product(product)
        exclusion = next((e for e in _excl if e in up), None)
        if exclusion:
            excluded = f'excluded ({exclusion})'
            continue
        for p in _pats:
            matches = (up == p[1:].strip()) if p.startswith('=') else (p in up)
            if matches:
                return True, p
    return False, excluded


seen = Counter()
matched_by = Counter()
for row in rows:
    prods = row.get('Assigned Products') or ''
    for part in _product_tokens(prods):
        seen[part] += 1
    ok, why = _is_copilot(prods)
    row['HasCopilot'] = 'TRUE' if ok else 'FALSE'
    if ok:
        matched_by[why] += 1

copilot_count = sum(1 for r in rows if r['HasCopilot'] == 'TRUE')
print(f'Total users          : {len(rows):,}')
print(f'With Copilot licence : {copilot_count:,}')

print('\nAssigned Products seen in this tenant:')
for prod, n in seen.most_common():
    up = prod.upper()
    hit, reason = _is_copilot(prod)
    tag = 'COUNTED' if hit else reason
    if ('COPILOT' in up or 'E7' in up) and not hit:
        tag = f'** Copilot/E7-like product NOT counted ** {reason}'
    print(f'    {n:>6}  {prod:<50} {tag}')

if matched_by:
    print('\nmatched by configured pattern (leading = means exact):')
    for p, n in matched_by.most_common():
        print(f'    {n:>6}  {p}')

_unmatched = [p for p in seen if ('COPILOT' in p.upper() or 'E7' in p.upper())
              and not _is_copilot(p)[0]]
if _unmatched:
    print('\n** REVIEW: these Copilot/E7-like products are NOT counted as licensed:')
    for p in _unmatched:
        print(f'      {p}')
    print('   Exclusions may be intentional. Verify entitlement before changing')
    print('   COPILOT_SKU_PATTERNS / COPILOT_SKU_EXCLUDE; do not match all Copilot or E7 names.')

if copilot_count == 0 and rows:
    print('\n** No qualifying assigned product matched. Review the snapshot,')
    print('   configured patterns and exclusions; zero may be legitimate.')

print('\nLicence flags reflect assigned products, not enabled/provisioned service plans.')


## 5. Build Spark DataFrame + canonicalise columns

Mirrors the existing CSV loader's column-renaming logic. UPN variants → `User Principal Name`, then sanitise spaces in column names (Delta forbids ` ,;{}()\n\t=`).


In [ ]:
import re
from pyspark.sql import functions as F

raw = spark.createDataFrame(rows)

# Detect UPN column variants (Graph returns 'User Principal Name' but be defensive)
upn_variants = ['User Principal Name', 'userPrincipalName', 'UserPrincipalName', 'User principal name']
actual_upn = next((c for c in upn_variants if c in raw.columns), None)
df = raw
if actual_upn and actual_upn != 'User Principal Name':
    df = df.withColumnRenamed(actual_upn, 'User Principal Name')

# Add canonical 'Has license' column from HasCopilot (alias to match existing loader output)
df = df.withColumnRenamed('HasCopilot', 'Has license')

# Add normalized join key + dedupe on it
df = df.withColumn(
    'UPN_Normalized',
    F.when(F.col('`User Principal Name`').isNull(), None)
     .otherwise(F.lower(F.trim(F.col('`User Principal Name`').cast('string'))))
)
df = df.dropDuplicates(['UPN_Normalized'])

# Sanitize column names (Delta forbids spaces / punctuation)
_INVALID = re.compile(r'[ ,;{}()\n\t=]')
df = df.toDF(*[_INVALID.sub('_', c) for c in df.columns])

print(f'After dedupe + sanitize: {df.count():,} rows')
print('Columns:', df.columns)


## 6. Write to Lakehouse Delta table

In [ ]:
(df.write
    .format('delta')
    .mode(WRITE_MODE)
    .option('overwriteSchema', 'true')
    .saveAsTable(OUTPUT_TABLE))

row_count = spark.table(OUTPUT_TABLE).count()
print(f'✓ Rows written to {OUTPUT_TABLE}: {row_count:,}')


## 7. Verify

In [ ]:
tbl = spark.table(OUTPUT_TABLE)
tbl.select('User_Principal_Name', 'Has_license', 'UPN_Normalized').show(10, truncate=False)
tbl.groupBy('Has_license').count().orderBy(F.desc('count')).show()


---
**Connect the PBIT**: this table is consumed by the `Copilot Licensed` query in both AI-in-One and ValueLens dashboards. Once this notebook has run, leave the `Copilot Licensed Users` parameter blank when opening the PBIT — refresh sources from `dbo.copilot_licensed_users` directly via the Fabric SQL endpoint.
